In [ ]:
from dag.artifact import Resources
from datasets.artifact import DataSet
from mappeddatasets.artifact import MappedDataSet
from models.mock.artifact import ModelParameters, Pretraining, PretrainingConfig
from sources.artifact import Source
from tokenizers.bpe import Tokenizer as BPETokenizer

In [5]:
odyssey = Source(name="odyssey", url="https://www.gutenberg.org/cache/epub/1727/pg1727.txt")
mobydick = Source(name="mobydick", url="https://www.gutenberg.org/cache/epub/2701/pg2701.txt")
romeojuliet = Source(name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt")
montecristo = Source(name="montecristo", url="https://www.gutenberg.org/cache/epub/1184/pg1184.txt")
pride = Source(name="pride", url="https://www.gutenberg.org/cache/epub/1342/pg1342.txt")
frankenstein = Source(name="frankenstein", url="https://www.gutenberg.org/cache/epub/84/pg84.txt")
greatexpectations = Source(name="greatexpectations", url="https://www.gutenberg.org/cache/epub/1400/pg1400.txt")
dracula = Source(name="dracula", url="https://www.gutenberg.org/cache/epub/345/pg345.txt")

In [7]:
RUN_ID = "RUN_V7"  # <- change this per run

# same tokenizer any other run trained on these sources would build -- if one's
# already declared, it's reused rather than rebuilt from scratch
tokenizer = BPETokenizer(
    vocab_size=1420,
    special_tokens=("<pad>", "<unk>"),
    sources=(odyssey, mobydick, dracula),
)

# also shared, same as the tokenizer above -- another run asking for
# this tokenizer and these exact sources reuses this train.bin/valid.bin
dataset = DataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=[montecristo, frankenstein],
    valid_sources=[pride,odyssey],
)

mappedset = MappedDataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=[montecristo, frankenstein],
    valid_sources=[dracula,pride],
)

model_parameters = ModelParameters(hidden_size=64, num_layers=2)
config = PretrainingConfig(
    total_steps=6000, batch_size=32, lr=1e-3, seed=1, checkpoint_every=500
)

pretraining = Pretraining(
    run_id=RUN_ID,
    dataset=mappedset,
    tokenizer=tokenizer,
    model_parameters=model_parameters,
    config=config,
    allocated_resources=Resources(gpu_type='T4', gpu_count=1)
)

pretraining  # parameters all the way down; `commit` is hidden from the repr

Pretraining(run_id='RUN_V7', dataset=MappedDataSet(train_set=(TokenizedSource(tokenizer=Tokenizer(vocab_size=1420, special_tokens=('<pad>', '<unk>'), sources=(Source(name='odyssey', url='https://www.gutenberg.org/cache/epub/1727/pg1727.txt'), Source(name='mobydick', url='https://www.gutenberg.org/cache/epub/2701/pg2701.txt'), Source(name='dracula', url='https://www.gutenberg.org/cache/epub/345/pg345.txt'))), source=Source(name='montecristo', url='https://www.gutenberg.org/cache/epub/1184/pg1184.txt')), TokenizedSource(tokenizer=Tokenizer(vocab_size=1420, special_tokens=('<pad>', '<unk>'), sources=(Source(name='odyssey', url='https://www.gutenberg.org/cache/epub/1727/pg1727.txt'), Source(name='mobydick', url='https://www.gutenberg.org/cache/epub/2701/pg2701.txt'), Source(name='dracula', url='https://www.gutenberg.org/cache/epub/345/pg345.txt'))), source=Source(name='frankenstein', url='https://www.gutenberg.org/cache/epub/84/pg84.txt'))), valid_set=(TokenizedSource(tokenizer=Tokenizer(v

In [ ]:
pretraining.declare()  # preview, no writes